# Roughness sensitivity analysis — HEC-RAS

Case study: **Besaya River — Corrales de Buelna**  
Model: HEC-RAS 6.6 2D (USACE), 5 m resolution, T=500 years

1,000 simulations with different Manning coefficient combinations are analysed.

**Bugs fixed relative to the original analysis:**
- TIFFs are loaded in numerical order (`load_flood_ensemble`) — not lexicographic.
- Spatial statistics use `mean(dim=['x','y'])` — not sequential.
- Flooded area uses factor `1e-6` to convert m² → km² — not `10e-6`.
- Wet threshold of 0.05 m applied at load time (consistent with SFINCS).

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from pyhydra.modeling.hydraulic.sensitivity import (
    load_flood_ensemble,
    compute_manning_stats,
    flooded_area,
    spatial_stats,
)

## Paths — adjust for your environment

In [2]:
import os
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
_candidates = [Path('/workspace'), _cwd, *_cwd.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / 'notebooks').exists() or (p / 'pyhydra').exists()),
    _cwd,
)
DATA_DIR  = Path(os.environ.get('HYDRA_DATA_DIR', str(REPO_ROOT / 'data')))
DATA_ROOT = DATA_DIR / 'pilot_cases' / 'manning_rugosidades' 
NB_DIR   = Path.cwd()
try:
    GEN_DIR  = DATA_ROOT / "generated"; GEN_DIR.mkdir(exist_ok=True)
except OSError:
    pass  # read-only filesystem in Docker

HECRAS_DIR       = DATA_ROOT / "hecras_results" / "results"
COMBINATIONS_DIR = DATA_ROOT / "combinaciones" / "nsim_rugos"
MANNING_TIF      = DATA_ROOT / "manning.tif"

CELL_AREA_M2 = 25.0
THRESHOLD    = 0.05

_ext_ok = HECRAS_DIR.exists() and any(HECRAS_DIR.glob("hamax_sim_*.tif"))
_has_comb = COMBINATIONS_DIR.exists() and any(COMBINATIONS_DIR.glob("combinacion_*.csv"))
_has_tif  = MANNING_TIF.exists()
_write_ok = True
try:
    (DATA_ROOT / ".write_test").touch()
    (DATA_ROOT / ".write_test").unlink()
except OSError:
    _write_ok = False
    print("⚠ Read-only filesystem — output files will not be saved")

## 1 · Load HEC-RAS results ensemble

In [3]:
if _ext_ok:
    flood = load_flood_ensemble(
        results_dir=str(HECRAS_DIR),
        pattern="hamax_sim_*.tif",
        threshold=THRESHOLD,
    )
    print(f"Ensemble HEC-RAS: {flood.dims} → shape {flood.shape}")
    print(f"Simulaciones: {flood.simulation.values[[0, 1, 2, -2, -1]]}")

## 2 · Load Manning roughness ensemble

In [4]:
if _ext_ok and _has_comb and _has_tif:
    sim_numbers = flood.simulation.values.tolist()
    try:
        manning_stats, reg, calado_stats = compute_manning_stats(
            raster_path=str(MANNING_TIF),
            combinations_dir=str(COMBINATIONS_DIR),
            flood_ensemble=flood,
            simulation_numbers=sim_numbers,
            cell_area_m2=CELL_AREA_M2,
            threshold=THRESHOLD,
        )
        print(f"Manning stats computed: {len(manning_stats)} simulations")
    except Exception as e:
        print(f"✗ compute_manning_stats failed: {type(e).__name__}: {e}")
        manning_stats = reg = calado_stats = None
elif _ext_ok and not _has_tif:
    print(f"⚠ Manning raster not found: {MANNING_TIF}")
    manning_stats = reg = calado_stats = None
elif _ext_ok:
    print('⚠ Combination CSVs not found — run notebook 01 first')
    manning_stats = reg = calado_stats = None

## 3 · Spatial statistics per simulation

In [5]:
if _ext_ok:
    if calado_stats is None:
        # Manning data unavailable — compute flood stats separately
        calado_stats = spatial_stats(flood)
        areas_m2 = flooded_area(flood, cell_area_m2=CELL_AREA_M2, threshold=THRESHOLD)
    else:
        areas_m2 = calado_stats["flooded_area_m2"].values

    print(calado_stats.describe().round(3))


## 4 · Output distributions across simulations

In [6]:
if _ext_ok and manning_stats is not None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    sns.histplot(areas_m2 * 1e-6, bins=25, kde=True, ax=axes[0])
    axes[0].set_xlabel("Flooded area (km²)")
    axes[0].set_title("Flooded area distribution")

    sns.histplot(manning_stats["median"], bins=25, kde=True, ax=axes[1])
    axes[1].set_xlabel("Spatial median of n Manning")
    axes[1].set_title("Median roughness distribution")

    sns.histplot(calado_stats["median"], bins=25, kde=True, ax=axes[2])
    axes[2].set_xlabel("Spatial median of water depth (m)")
    axes[2].set_title("Median water depth distribution")

    plt.suptitle("HEC-RAS — 1,000 simulations", y=1.01)
    plt.tight_layout()
    plt.show()

## 5 · Roughness ↔ water depth and flooded area regression

In [7]:
if _ext_ok and reg is not None:
    reg.head()

In [8]:
if _ext_ok and reg is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.scatter(reg["manning_mean"], reg["depth_mean"], alpha=0.4, s=10)
    coef = np.polyfit(reg["manning_mean"].dropna(), reg["depth_mean"].dropna(), 1)
    x_line = np.linspace(reg["manning_mean"].min(), reg["manning_mean"].max(), 100)
    ax.plot(x_line, np.polyval(coef, x_line), "r-",
            label=f"y = {coef[0]:.2f}x + {coef[1]:.2f}")
    ax.set_xlabel("Mean n Manning (wet cells)")
    ax.set_ylabel("Mean water depth (m)")
    ax.set_title("Manning vs Water Depth — HEC-RAS")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.scatter(reg["manning_mean"], reg["flooded_area_m2"] * 1e-6, alpha=0.4, s=10)
    valid = reg[["manning_mean", "flooded_area_m2"]].dropna()
    coef2 = np.polyfit(valid["manning_mean"], valid["flooded_area_m2"] * 1e-6, 1)
    ax.plot(x_line, np.polyval(coef2, x_line), "r-",
            label=f"y = {coef2[0]:.2f}x + {coef2[1]:.2f}")
    ax.set_xlabel("Mean n Manning (wet cells)")
    ax.set_ylabel("Flooded area (km²)")
    ax.set_title("Manning vs Flooded Area — HEC-RAS")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 6 · 3D scatter: depth × area × roughness

In [9]:
if _ext_ok and reg is not None:
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection="3d")
    sc = ax.scatter(
        reg["depth_mean"],
        reg["flooded_area_m2"] * 1e-6,
        reg["manning_mean"],
        c=reg["manning_mean"],
        cmap="viridis",
        alpha=0.6,
        s=15,
    )
    ax.set_xlabel("Mean water depth (m)")
    ax.set_ylabel("Flooded area (km²)")
    ax.set_zlabel("Mean n Manning")
    ax.set_title("HEC-RAS — depth / area / roughness relationship (1,000 sims)")
    fig.colorbar(sc, ax=ax, label="n Manning", shrink=0.5)
    plt.tight_layout()
    plt.show()

## 7 · Simulation with highest mean water depth

In [10]:
if _ext_ok:
    sim_max = int(calado_stats["mean"].idxmax())
    print(f"Simulation with highest mean water depth: sim_{sim_max}")
    print(calado_stats.loc[sim_max].round(3))

    fig, ax = plt.subplots(figsize=(8, 6))
    flood.sel(simulation=sim_max).plot(ax=ax, cmap="Blues", vmin=0, vmax=5)
    ax.set_title(f"Maximum water depth — sim_{sim_max} (highest n Manning)")
    plt.tight_layout()
    plt.show()

## 8 · Save results table

In [11]:
if _write_ok:
    if _ext_ok:
        results = reg.copy()
        results["flooded_area_km2"] = results["flooded_area_m2"] * 1e-6
        results.drop(columns="flooded_area_m2", inplace=True)

        out_csv = GEN_DIR / "hecras_sensitivity_results.csv"
        results.to_csv(out_csv)
        print(f"Guardado: {out_csv.resolve()}")